# 🧠 Brain Tumor Segmentation Testing

This notebook evaluates trained segmentation models on test data.

**Features:**
- Load trained models
- Evaluate on test/validation data
- Visualize predictions with overlays
- Calculate segmentation metrics (Dice, IoU, Sensitivity, Specificity)

## 1. Setup

In [ ]:
import os
import sys

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Import Segmentation Module

In [ ]:
from src.segmentation import TumorSegmentor
from src.segmentation.training.metrics import (
    dice_coefficient,
    iou_score,
    sensitivity,
    specificity,
    precision_metric
)

print("✅ Modules imported successfully!")

## 3. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TEST CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# Model to test
MODEL_PATH = "../weights/segmentation/UNet_bce_tversky_best.keras"
MODEL_NAME = "UNet"  # For visualization titles
LOSS_NAME = "bce_tversky"

# Data paths (use validation or test set)
TEST_IMAGES_DIR = "../data/brisc2025/segmentation_task/test/images"  # Change to test if available
TEST_MASKS_DIR = "../data/brisc2025/segmentation_task/test/masks"

# Settings
IMG_SIZE = (256, 256)
BATCH_SIZE = 16
THRESHOLD = 0.5
NUM_SAMPLES = 8  # Number of samples to visualize

# Output directory
OUTPUT_DIR = "../logs/segmentation/test_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Configuration set!")

## 4. Load Model

In [ ]:
# Initialize segmentor with trained model
segmentor = TumorSegmentor(
    model_path=MODEL_PATH,
    img_size=IMG_SIZE,
    threshold=THRESHOLD,
)

print(f"✅ Model loaded: {MODEL_PATH}")
print(f"   Model summary:")
segmentor.model.summary()

## 5. Prepare Test Data

In [ ]:
from src.segmentation.data import create_segmentation_datasets

# Create dataset (using validation split for testing)
train_ds, val_ds, _ = create_segmentation_datasets(
    train_images_dir=TEST_IMAGES_DIR,
    train_masks_dir=TEST_MASKS_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_split=0.2,
    random_state=42,
    use_augmentation=False,  # No augmentation for testing
)

# Use validation set for testing
test_ds = val_ds

print(f"✅ Test dataset prepared")

## 6. Evaluate Model

In [ ]:
# Evaluate on test set
print("\n📊 Evaluating model on test set...\n")

results = segmentor.model.evaluate(test_ds, verbose=1)

# Print results
metric_names = ["Loss"] + [m.name for m in segmentor.model.metrics]
results_dict = dict(zip(metric_names[:len(results)], results))

print("\n" + "="*60)
print(f"📊 Test Results for {MODEL_NAME}")
print("="*60)
for metric, value in results_dict.items():
    print(f"{metric:.<30} {value:.4f}")
print("="*60)

## 7. Visualize Sample Predictions

In [ ]:
# Get sample batch
for images, masks in test_ds.take(1):
    # Limit to NUM_SAMPLES
    images = images[:NUM_SAMPLES]
    masks = masks[:NUM_SAMPLES]
    
    # Predict
    predictions = segmentor.model.predict(images, verbose=0)
    
    # Create visualization
    fig, axes = plt.subplots(NUM_SAMPLES, 4, figsize=(16, 4*NUM_SAMPLES))
    
    for i in range(NUM_SAMPLES):
        # Original Image
        axes[i, 0].imshow(images[i].numpy())
        axes[i, 0].set_title('Original Image', fontsize=12, fontweight='bold')
        axes[i, 0].axis('off')
        
        # Ground Truth Mask
        axes[i, 1].imshow(masks[i].numpy()[:, :, 0], cmap='hot')
        axes[i, 1].set_title('Ground Truth', fontsize=12, fontweight='bold')
        axes[i, 1].axis('off')
        
        # Predicted Mask
        axes[i, 2].imshow(predictions[i][:, :, 0], cmap='hot')
        axes[i, 2].set_title('Prediction', fontsize=12, fontweight='bold')
        axes[i, 2].axis('off')
        
        # Overlay on Original
        overlay = segmentor.overlay_mask(
            images[i].numpy(),
            predictions[i],
            color=(255, 0, 0),
            alpha=0.4
        )
        axes[i, 3].imshow(overlay)
        axes[i, 3].set_title('Overlay', fontsize=12, fontweight='bold')
        axes[i, 3].axis('off')
    
    plt.suptitle(f'{MODEL_NAME} - Sample Predictions', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    # Save
    save_path = f"{OUTPUT_DIR}/{MODEL_NAME}_{LOSS_NAME}_sample_predictions.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"\n💾 Saved visualization: {save_path}")
    
    plt.show()

## 8. Calculate Per-Image Metrics

In [ ]:
# Calculate metrics for individual images
dice_scores = []
iou_scores = []
sensitivity_scores = []
specificity_scores = []

print("\n📊 Calculating per-image metrics...")

for images, masks in test_ds:
    predictions = segmentor.model.predict(images, verbose=0)
    
    for i in range(len(images)):
        y_true = tf.expand_dims(masks[i], axis=0)
        y_pred = tf.expand_dims(predictions[i], axis=0)
        
        dice_scores.append(dice_coefficient(y_true, y_pred).numpy())
        iou_scores.append(iou_score(y_true, y_pred).numpy())
        sensitivity_scores.append(sensitivity(y_true, y_pred).numpy())
        specificity_scores.append(specificity(y_true, y_pred).numpy())

# Calculate statistics
print("\n" + "="*60)
print("📊 Per-Image Metrics Statistics")
print("="*60)
print(f"Dice Coefficient:")
print(f"  Mean: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"  Min:  {np.min(dice_scores):.4f} | Max: {np.max(dice_scores):.4f}")
print()
print(f"IoU Score:")
print(f"  Mean: {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")
print(f"  Min:  {np.min(iou_scores):.4f} | Max: {np.max(iou_scores):.4f}")
print()
print(f"Sensitivity:")
print(f"  Mean: {np.mean(sensitivity_scores):.4f} ± {np.std(sensitivity_scores):.4f}")
print(f"  Min:  {np.min(sensitivity_scores):.4f} | Max: {np.max(sensitivity_scores):.4f}")
print()
print(f"Specificity:")
print(f"  Mean: {np.mean(specificity_scores):.4f} ± {np.std(specificity_scores):.4f}")
print(f"  Min:  {np.min(specificity_scores):.4f} | Max: {np.max(specificity_scores):.4f}")
print("="*60)

## 9. Metrics Distribution

In [ ]:
# Plot metrics distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Dice Coefficient
axes[0, 0].hist(dice_scores, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(np.mean(dice_scores), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(dice_scores):.3f}')
axes[0, 0].set_title('Dice Coefficient Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Dice Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# IoU Score
axes[0, 1].hist(iou_scores, bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(np.mean(iou_scores), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(iou_scores):.3f}')
axes[0, 1].set_title('IoU Score Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('IoU Score')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Sensitivity
axes[1, 0].hist(sensitivity_scores, bins=30, color='salmon', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(np.mean(sensitivity_scores), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(sensitivity_scores):.3f}')
axes[1, 0].set_title('Sensitivity Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Sensitivity')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Specificity
axes[1, 1].hist(specificity_scores, bins=30, color='plum', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(np.mean(specificity_scores), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(specificity_scores):.3f}')
axes[1, 1].set_title('Specificity Distribution', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Specificity')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'{MODEL_NAME} - Metrics Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()

# Save
save_path = f"{OUTPUT_DIR}/{MODEL_NAME}_{LOSS_NAME}_metrics_distribution.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"\n💾 Saved metrics distribution: {save_path}")

plt.show()

## 10. Summary Report

In [ ]:
print("\n" + "="*60)
print(f"🎯 FINAL SUMMARY - {MODEL_NAME} ({LOSS_NAME})")
print("="*60)
print(f"Model Path: {MODEL_PATH}")
print(f"Test Images: {len(dice_scores)}")
print(f"Image Size: {IMG_SIZE}")
print(f"Threshold: {THRESHOLD}")
print()
print("Performance Metrics:")
print(f"  Dice Coefficient: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"  IoU Score:        {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")
print(f"  Sensitivity:      {np.mean(sensitivity_scores):.4f} ± {np.std(sensitivity_scores):.4f}")
print(f"  Specificity:      {np.mean(specificity_scores):.4f} ± {np.std(specificity_scores):.4f}")
print()
print(f"Output Directory: {OUTPUT_DIR}")
print("="*60)
print("\n✅ Testing complete!")